# Fase 4 — Interface

**Projeto CBL — Sistemas de Machine Learning**
**Fase:** Act (implementação da Seção 8 de `../docs/planejamentoModelo.md`)

Este notebook implementa o "motor" por trás da interface: uma função única de ponta a ponta (`gerar_playlist`) que combina a seleção de candidatas (Fase 2) e o sequenciamento por transição suave (Fase 3), e usa esse motor para pré-computar playlists reais para todas as combinações de 1 a 3 palavras do vocabulário de mood — dados que alimentam a interface visual (artifact HTML, publicado separadamente).

Consome: `df_clean.parquet`, `catalogo_com_mood.parquet`, `modelo_mood.joblib` (Fase 2).

In [1]:
import pandas as pd
import numpy as np
import json
import joblib
from itertools import combinations
from scipy.spatial.distance import cdist

modelo = joblib.load("modelo_mood.joblib")
pt, scaler, gmm, knn = modelo['power_transformer'], modelo['scaler'], modelo['gmm'], modelo['knn']
features, cols_assimetricas = modelo['features'], modelo['cols_assimetricas']
vocab_df, vocab_X = modelo['vocab_df'], modelo['vocab_X']

df_clean = pd.read_parquet("df_clean.parquet")
mood_df = pd.read_parquet("catalogo_com_mood.parquet")
assert (df_clean['track_id'].values == mood_df['track_id'].values).all(), "ordem das linhas não bate"
df = df_clean.copy()
df['mood'] = mood_df['mood'].values

X_raw = df[features].copy()
X_raw[cols_assimetricas] = pt.transform(X_raw[cols_assimetricas])
X = scaler.transform(X_raw[features])

print(f"Catálogo carregado: {len(df)} faixas")
print(f"Vocabulário: {list(vocab_df.index)}")

Catálogo carregado: 113549 faixas
Vocabulário: ['Energetico', 'Feliz', 'Dancante', 'Calmo', 'Melancolico', 'Intenso', 'Instrumental', 'Acustico']


## 1. Motor: `gerar_playlist`

Reaproveita as funções já validadas nas Fases 2 e 3 (Camelot Wheel, seleção de candidatas com deduplicação por `(track_name, artists)`, sequenciamento por transição suave), combinadas numa única chamada — a função que uma interface real (chat ou tela dedicada) chamaria a cada pedido do usuário.

In [2]:
CAMELOT_MAIOR = {0:8, 1:3, 2:10, 3:5, 4:12, 5:7, 6:2, 7:9, 8:4, 9:11, 10:6, 11:1}
CAMELOT_MENOR = {0:5, 1:12, 2:7, 3:2, 4:9, 5:4, 6:11, 7:6, 8:1, 9:8, 10:3, 11:10}

def camelot(key, mode):
    numero = CAMELOT_MAIOR[key] if mode == 1 else CAMELOT_MENOR[key]
    return (numero, 'B' if mode == 1 else 'A')

def penalidade_harmonica(a, b):
    (na, la), (nb, lb) = a, b
    if na == nb and la == lb: return 0.0
    if na == nb and la != lb: return 0.15
    diferenca = min((na - nb) % 12, (nb - na) % 12)
    if diferenca == 1 and la == lb: return 0.15
    return 0.5

df['camelot'] = df.apply(lambda r: camelot(int(r['key']), int(r['mode'])), axis=1)


def selecionar_candidatas(palavras, n=20):
    """Busca vizinhos em excesso e deduplica por (track_name, artists) — necessário porque
    o catálogo tem duplicatas por track_id repetido em gênero (Passo 2.2 do EDA) e músicas
    reeditadas sob track_id diferente (~9,4% do catálogo, achado da Fase 3). Se o multiplicador
    inicial não trouxer candidatas únicas suficientes (regiões muito densas em duplicata), a
    busca escalona (dobra o raio) em vez de silenciosamente devolver menos faixas que o pedido."""
    indices_ancoras = [vocab_df.index.get_loc(p) for p in palavras]
    alvo = vocab_X[indices_ancoras].mean(axis=0).reshape(1, -1)

    multiplicador = 6
    while True:
        k = min(n * multiplicador, len(df))
        distancias, indices = knn.kneighbors(alvo, n_neighbors=k)
        vistos = set()
        idx_finais = []
        for idx_faixa in indices[0]:
            linha = df.iloc[idx_faixa]
            chave = (linha['track_name'], linha['artists'])
            if chave not in vistos:
                vistos.add(chave)
                idx_finais.append(idx_faixa)
            if len(idx_finais) == n:
                break
        if len(idx_finais) == n or k >= len(df):
            break
        multiplicador *= 2

    resultado = df.iloc[idx_finais].reset_index(drop=True)
    return resultado, X[idx_finais]


def sequenciar(candidatas, X_candidatas, peso_tempo=0.5, peso_harmonico=1.0):
    n = len(candidatas)
    custo = cdist(X_candidatas, X_candidatas)
    tempos = candidatas['tempo'].values
    dist_tempo = np.abs(tempos[:, None] - tempos[None, :]) / df['tempo'].std()
    camelots = candidatas['camelot'].tolist()
    for i in range(n):
        for j in range(n):
            if i != j:
                custo[i, j] += peso_tempo * dist_tempo[i, j] + peso_harmonico * penalidade_harmonica(camelots[i], camelots[j])
    visitado, atual, restante = [0], 0, set(range(1, n))
    while restante:
        proximo = min(restante, key=lambda j: custo[atual, j])
        visitado.append(proximo); restante.remove(proximo); atual = proximo
    return candidatas.iloc[visitado].reset_index(drop=True)


def gerar_playlist(palavras, n=20):
    """Função de ponta a ponta: palavras de mood -> playlist sequenciada."""
    candidatas, X_cand = selecionar_candidatas(palavras, n=n)
    return sequenciar(candidatas, X_cand)


# Teste rápido
gerar_playlist(['Calmo', 'Instrumental'], n=10)[['track_name', 'artists', 'tempo', 'mood']]

,track_name,artists,tempo,mood
0,Give a Shit,Greensky Bluegrass,106.510,Calmo
1,Maggie May,Rod Stewart,129.337,Calmo
2,Pelangi Baruku,Dhyo Haw,137.031,Instrumental
3,Aaro Nenjil - Desi Mix,Gowry Lekshmi,130.046,Instrumental
4,Voltar pra Casa,Gloria,119.982,Calmo
5,Screams,Greensky Bluegrass,114.082,Calmo
6,Heimat mein,Fäaschtbänkler,100.046,Calmo
7,I'll Take A Melody - Live,Jerry Garcia Band;Jerry Garcia,122.701,Instrumental
8,Paradise,Satanicpornocultshop,169.936,Instrumental
9,You're the Sea,Andrew Belle,161.025,Instrumental


## 2. Pré-computação para a interface

A interface visual não tem acesso a um backend Python em tempo real — é uma página estática. A solução (já usada e justificada na avaliação da API do Spotify) é **pré-calcular offline**.

Duas dimensões precisam ser cobertas: a combinação de palavras de mood (1 a 3, `C(8,1)+C(8,2)+C(8,3) = 92`) **e** o tamanho da playlist — a interface anterior fixava `n=20`; agora o usuário escolhe entre `{10, 15, 20, 25, 30}` faixas. Total: `92 × 5 = 460` playlists pré-computadas.

In [3]:
TAMANHOS_PLAYLIST = [10, 15, 20, 25, 30]

palavras_vocab = list(vocab_df.index)
todas_combinacoes = []
for tamanho in [1, 2, 3]:
    todas_combinacoes.extend(combinations(palavras_vocab, tamanho))

print(f"Combinações de mood: {len(todas_combinacoes)} x tamanhos: {len(TAMANHOS_PLAYLIST)} = {len(todas_combinacoes)*len(TAMANHOS_PLAYLIST)} playlists")

resultado_precomputado = {}
for combo in todas_combinacoes:
    for n in TAMANHOS_PLAYLIST:
        playlist = gerar_playlist(list(combo), n=n)
        chave = "+".join(combo) + f"|{n}"
        resultado_precomputado[chave] = (
            playlist.assign(camelot_str=playlist['camelot'].apply(lambda c: f"{c[0]}{c[1]}"))
                    [['track_name', 'artists', 'tempo', 'track_genre', 'mood', 'camelot_str']]
                    .rename(columns={'camelot_str': 'camelot'})
                    .to_dict('records')
        )

print(f"Playlists pré-computadas: {len(resultado_precomputado)}")
print(f"Exemplo de chave: {[k for k in resultado_precomputado if k.startswith('Calmo+Instrumental')]}")

Combinações de mood: 92 x tamanhos: 5 = 460 playlists


Playlists pré-computadas: 460
Exemplo de chave: ['Calmo+Instrumental|10', 'Calmo+Instrumental|15', 'Calmo+Instrumental|20', 'Calmo+Instrumental|25', 'Calmo+Instrumental|30', 'Calmo+Instrumental+Acustico|10', 'Calmo+Instrumental+Acustico|15', 'Calmo+Instrumental+Acustico|20', 'Calmo+Instrumental+Acustico|25', 'Calmo+Instrumental+Acustico|30']


## 3. Artefato final: dados para a interface

In [4]:
with open("playlists_precomputadas.json", "w", encoding="utf-8") as f:
    json.dump(resultado_precomputado, f, ensure_ascii=False)

import os
tamanho_kb = os.path.getsize("playlists_precomputadas.json") / 1024
print(f"Salvo: entregas/playlists_precomputadas.json ({tamanho_kb:.0f} KB, {len(resultado_precomputado)} playlists pré-computadas)")

Salvo: entregas/playlists_precomputadas.json (1348 KB, 460 playlists pré-computadas)


## 4. Links para escuta (validação — Fase 5)

Não é possível criar uma playlist de verdade salva numa conta do YouTube sem autenticação OAuth com uma conta Google real — não temos essa credencial neste ambiente. O que dá pra fazer sem nenhuma autenticação: gerar, para cada faixa, um **link de busca do YouTube** (`youtube.com/results?search_query=...`) — abre a busca já com o nome certo, na ordem sequenciada da playlist. É o mecanismo usado na interface (cada faixa vira um link clicável).

In [5]:
from urllib.parse import quote_plus

def link_youtube(track_name, artists):
    consulta = f"{track_name} {artists}"
    return f"https://www.youtube.com/results?search_query={quote_plus(consulta)}"

def playlist_para_escuta(palavras, n=15):
    """Gera a playlist e imprime, faixa a faixa, o link de busca no YouTube na ordem sequenciada —
    para uso direto na validação por escuta (Fase 5), sem precisar abrir a interface."""
    playlist = gerar_playlist(palavras, n=n)
    for i, row in playlist.iterrows():
        print(f"{i+1:2d}. {row['track_name']} — {row['artists']}  ({row['tempo']:.0f} BPM)")
        print(f"    {link_youtube(row['track_name'], row['artists'])}")
    return playlist

_ = playlist_para_escuta(['Feliz', 'Dancante'], n=8)

 1. Sin Sentimiento — Grupo Galé  (172 BPM)
    https://www.youtube.com/results?search_query=Sin+Sentimiento+Grupo+Gal%C3%A9
 2. What Would You Do? — Joel Corry;David Guetta;Bryson Tiller  (124 BPM)
    https://www.youtube.com/results?search_query=What+Would+You+Do%3F+Joel+Corry%3BDavid+Guetta%3BBryson+Tiller
 3. Firework — Katy Perry  (124 BPM)
    https://www.youtube.com/results?search_query=Firework+Katy+Perry
 4. 共犯者 - Remastered 2022 — Eikichi Yazawa  (110 BPM)
    https://www.youtube.com/results?search_query=%E5%85%B1%E7%8A%AF%E8%80%85+-+Remastered+2022+Eikichi+Yazawa
 5. 乗り遅れたバス — Keyakizaka46  (112 BPM)
    https://www.youtube.com/results?search_query=%E4%B9%97%E3%82%8A%E9%81%85%E3%82%8C%E3%81%9F%E3%83%90%E3%82%B9+Keyakizaka46
 6. Me Vuelvo Loco — Abraham Mateo;CNCO  (92 BPM)
    https://www.youtube.com/results?search_query=Me+Vuelvo+Loco+Abraham+Mateo%3BCNCO
 7. Happier — Marshmello;Bastille  (100 BPM)
    https://www.youtube.com/results?search_query=Happier+Marshmello%3BBasti

## Conclusão da Fase 4

- `gerar_playlist(palavras, n)` — motor de ponta a ponta, combinando Fase 2 (seleção de candidatas) + Fase 3 (sequenciamento).
- 460 playlists pré-computadas (92 combinações de mood × 5 tamanhos: 10/15/20/25/30 faixas) exportadas em `playlists_precomputadas.json` — a interface visual consome esse arquivo estático.
- `link_youtube` / `playlist_para_escuta` — geram links de busca do YouTube por faixa, na ordem sequenciada, para validação por escuta (Fase 5). Não cria uma playlist real salva numa conta (exigiria OAuth com credenciais que não temos), mas permite ouvir a sequência completa clicando link a link.
- Consistente com a decisão já registrada na avaliação da API do Spotify: pré-cálculo offline é a arquitetura correta para este caso de uso, não computação em tempo real.